# OpenPlaque — R1 RCA continuity test (self-contained)

This notebook starts from `main` and contains the complete experiment in visible cells. It does not `%run` an external Python file.

It loads source CCTA series 7, loads or creates the TotalSegmentator aorta mask, localizes the aortic root, rediscovers the best RCA-like ostium neighborhood, and tests whether a coronary-sized high-contrast path continues 10–20 mm away from the aortic wall.

No full RCA centerline is accepted here; this is a local continuity validation.


In [ ]:
# FIRST EXECUTABLE CELL: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch rca-r1-continuity-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque

%pip -q install --upgrade \
  "git+https://github.com/MIC-DKFZ/nnUNet.git" \
  "git+https://github.com/wasserth/TotalSegmentator.git" \
  pydicom SimpleITK nibabel scipy scikit-image matplotlib pandas
print('Packages installed.')


In [ ]:
import os, sys, time, shutil, subprocess, inspect
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, SimpleITK as sitk
from scipy import ndimage as ndi
from skimage.filters import frangi
from skimage.graph import route_through_array

SRC=Path('/content/OpenPlaque/src')
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))
from openplaque.study import OpenPlaqueStudy

ROOT=Path('/content/drive/MyDrive/OpenPlaque')
OUTDIR=ROOT/'RCA_R1_Continuity'
OUTDIR.mkdir(parents=True,exist_ok=True)


## 1. Load best-diastolic source CCTA series 7


In [ ]:
DRIVE_ZIP=ROOT/'Full_DICOM.zip'; LOCAL_ZIP=Path('/content/Full_DICOM.zip')
EXTRACT_ROOT='/content/full_dicom_r1_continuity'
if not DRIVE_ZIP.exists(): raise FileNotFoundError(DRIVE_ZIP)
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size!=DRIVE_ZIP.stat().st_size:
    print('Copying Full_DICOM.zip locally...',flush=True); shutil.copyfile(DRIVE_ZIP,LOCAL_ZIP)
shutil.rmtree(EXTRACT_ROOT,ignore_errors=True)
study=OpenPlaqueStudy(str(LOCAL_ZIP),extract_root=EXTRACT_ROOT)
source_img,source,_=study.load_series(7); source=np.asarray(source)
spacing_xyz=np.array(source_img.GetSpacing(),float); spacing_zyx=spacing_xyz[::-1]
origin=np.array(source_img.GetOrigin(),float); direction=np.array(source_img.GetDirection(),float).reshape(3,3)
voxel_mm3=float(np.prod(spacing_xyz))
print('shape z,y,x:',source.shape,' spacing x,y,z:',tuple(spacing_xyz))
INPUT_NII=Path('/content/source_series7.nii.gz'); sitk.WriteImage(source_img,str(INPUT_NII))


## 2. Load cached TotalSegmentator aorta mask; run TotalSegmentator only if the cache is absent


In [ ]:
def geometry_matches(a,b):
    return (a.GetSize()==b.GetSize() and np.allclose(a.GetSpacing(),b.GetSpacing(),atol=1e-5)
            and np.allclose(a.GetOrigin(),b.GetOrigin(),atol=1e-4)
            and np.allclose(a.GetDirection(),b.GetDirection(),atol=1e-5))
def read_mask(path):
    img=sitk.ReadImage(str(path))
    if not geometry_matches(img,source_img):
        img=sitk.Resample(img,source_img,sitk.Transform(),sitk.sitkNearestNeighbor,0,sitk.sitkUInt8)
    return sitk.GetArrayFromImage(img)>0

cache=ROOT/'RCA_Ostium_TotalSegmentator'/'aorta_series7_totalseg.nii.gz'
if not cache.exists():
    import torch, nnunetv2.inference.data_iterators as di
    p=Path(inspect.getfile(di)); txt=p.read_text()
    patched=txt.replace('manager = Manager()','manager = context.Manager()')
    patched=patched.replace("multiprocessing.get_context('spawn')","multiprocessing.get_context('fork')")
    patched=patched.replace('multiprocessing.get_context("spawn")','multiprocessing.get_context("fork")')
    if patched!=txt: p.write_text(patched)
    work=Path('/content/totalseg_aorta_r1'); shutil.rmtree(work,ignore_errors=True); work.mkdir()
    dev='gpu' if torch.cuda.is_available() else 'cpu'
    cmd=['TotalSegmentator','-i',str(INPUT_NII),'-o',str(work),'-ta','total','-rs','aorta','--device',dev,'-nr','1','-ns','1']
    rc=subprocess.run(cmd).returncode
    if rc: raise RuntimeError('TotalSegmentator aorta segmentation failed')
    prod=work/'aorta.nii.gz'
    cache.parent.mkdir(parents=True,exist_ok=True); shutil.copyfile(prod,cache)
aorta=read_mask(cache)
print('Aorta cache:',cache,' volume mL:',round(aorta.sum()*voxel_mm3/1000,2))


## 3. Localize the ascending-aortic root from the aorta mask in LPS coordinates


In [ ]:
def index_xyz_to_lps(x,y,z):
    return origin + direction @ (np.array([x,y,z],float)*spacing_xyz)
def array_zyx_to_lps(q):
    z,y,x=map(float,q); return index_xyz_to_lps(x,y,z)

pix_area=float(spacing_xyz[0]*spacing_xyz[1]); rows=[]
for z in range(aorta.shape[0]):
    lab,n=ndi.label(aorta[z],structure=np.ones((3,3),bool)); comps=[]
    for k in range(1,n+1):
        yy,xx=np.where(lab==k)
        if not len(xx): continue
        area=len(xx)*pix_area
        if not (120<=area<=3000): continue
        cy,cx=float(np.mean(yy)),float(np.mean(xx)); lps=index_xyz_to_lps(cx,cy,z)
        comps.append(dict(label=k,area_mm2=area,r_mm=np.sqrt(area/np.pi),cy=cy,cx=cx,
                          lps_x=lps[0],lps_y=lps[1],lps_z=lps[2]))
    if len(comps)>=2:
        ant=min(comps,key=lambda c:c['lps_y']); post=max(comps,key=lambda c:c['lps_y'])
        sep=post['lps_y']-ant['lps_y']
        if 9<=ant['r_mm']<=28 and sep>=12:
            rows.append({'z':z,**{f'a_{k}':v for k,v in ant.items()},'ap_sep_mm':sep})
track=pd.DataFrame(rows).sort_values('z')
if track.empty: raise RuntimeError('Could not identify ascending-aorta slices')
runs=[]; cur=[int(track.iloc[0].z)]
for z in track.z.astype(int).tolist()[1:]:
    if z-cur[-1]<=4: cur.append(z)
    else: runs.append(cur); cur=[z]
runs.append(cur)
def score_run(r):
    s=track[track.z.isin(r)]; return abs(float(s.a_lps_z.max()-s.a_lps_z.min()))+.05*len(r)
run=max(runs,key=score_run); run_df=track[track.z.isin(run)].sort_values('a_lps_z')
smin=float(run_df.a_lps_z.min())
root_df=run_df[(run_df.a_lps_z>=smin+1.5)&(run_df.a_lps_z<=smin+25.5)].copy()
if len(root_df)<8:
    qlo,qhi=run_df.a_lps_z.quantile([.02,.35]); root_df=run_df[(run_df.a_lps_z>=qlo)&(run_df.a_lps_z<=qhi)].copy()
root_zs=sorted(root_df.z.astype(int).unique())
asc=np.zeros_like(aorta,bool); centers={}
for _,r in root_df.iterrows():
    z=int(r.z); lab,_=ndi.label(aorta[z],structure=np.ones((3,3),bool))
    asc[z]=(lab==int(r.a_label)); centers[z]=np.array([r.a_lps_x,r.a_lps_y,r.a_lps_z])
print('Root z range:',min(root_zs),'to',max(root_zs))


In [ ]:
show=np.linspace(min(root_zs),max(root_zs),min(12,len(root_zs))).round().astype(int)
show=[min(root_zs,key=lambda q:abs(q-int(z))) for z in show]; show=list(dict.fromkeys(show))
fig,axs=plt.subplots(3,4,figsize=(16,12))
for ax in axs.ravel(): ax.axis('off')
for ax,z in zip(axs.ravel(),show):
    ax.imshow(source[z],cmap='gray',vmin=-200,vmax=900)
    ax.contour(aorta[z].astype(float),levels=[.5],linewidths=1)
    ax.contour(asc[z].astype(float),levels=[.5],linewidths=2)
    ax.set_title(f'root band z={z}'); ax.axis('off')
plt.tight_layout(); p=OUTDIR/'01_root_band_validation.png'; fig.savefig(p,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
print('Saved:',p)


## 4. Rediscover the best RCA-like ostium neighborhood automatically


In [ ]:
med_hu=float(np.median(source[asc])); blood_thr=float(np.clip(.43*med_hu,180,360))
dist=ndi.distance_transform_edt(~asc,sampling=spacing_zyx)
nb=(dist<=22)&(dist>.9)
slice_s=np.array([index_xyz_to_lps(0,0,z)[2] for z in range(source.shape[0])])
slo=float(root_df.a_lps_z.min())-5; shi=float(root_df.a_lps_z.max())+5
nb &= ((slice_s>=min(slo,shi))&(slice_s<=max(slo,shi)))[:,None,None]
blood=(source>=blood_thr)&nb
blood=ndi.binary_closing(blood,structure=np.ones((3,3,3),bool),iterations=1)
lab3,n3=ndi.label(blood,structure=np.ones((3,3,3),bool))
rows=[]
for k in range(1,n3+1):
    c=np.argwhere(lab3==k)
    if len(c)<5: continue
    vol=len(c)*voxel_mm3
    if vol>5000: continue
    dv=dist[tuple(c.T)]; mind,maxd=float(dv.min()),float(dv.max())
    if mind>3 or maxd<1.8: continue
    mm=c.astype(float)*spacing_zyx; ev=np.linalg.eigvalsh(np.cov(mm,rowvar=False)) if len(mm)>=3 else np.array([1e-6]*3)
    ev=np.sort(np.maximum(ev,1e-6))[::-1]; L=float(np.sqrt(12*ev[0])); elong=float(ev[0]/max(ev[1],1e-6))
    ns=int(np.unique(c[:,0]).size)
    z0,y0,x0=c.min(0); z1,y1,x1=c.max(0)+1; sub=(lab3[z0:z1,y0:y1,x0:x1]==k)
    rmax=float(ndi.distance_transform_edt(sub,sampling=spacing_zyx).max())
    near=c[dv<=mind+max(.8,float(min(spacing_xyz))*2)]; contact=np.median(near,axis=0); lps=array_zyx_to_lps(contact)
    zr=min(root_zs,key=lambda q:abs(q-int(round(contact[0])))); ac=centers[zr]
    dx=float(lps[0]-ac[0]); dy=float(lps[1]-ac[1])
    base=.22*np.tanh(L/10)+.20*np.tanh(maxd/8)+.18*np.tanh(elong/5)+.14*np.tanh(ns/8)+.12*np.exp(-mind/1.5)+.14*np.exp(-((rmax-2.2)/2.2)**2)
    score=float(base+.12*np.tanh(max(0,-dx)/8)+.04*np.tanh(max(0,-dy)/8))
    rows.append(dict(label=k,score=score,z_contact=contact[0],y_contact=contact[1],x_contact=contact[2],
                     lps_x=lps[0],lps_y=lps[1],lps_z=lps[2],dx_from_aorta_mm=dx,dy_from_aorta_mm=dy,
                     volume_mm3=vol,length_est_mm=L,elongation=elong,slices=ns,min_dist_aorta_mm=mind,
                     max_dist_aorta_mm=maxd,max_radius_mm=rmax,mean_hu=float(np.mean(source[tuple(c.T)]))))
cand=pd.DataFrame(rows)
pool=cand[(cand.dx_from_aorta_mm<1.5)&(cand.max_radius_mm<=5.5)&(cand.length_est_mm>=2)&(cand.slices>=2)].copy()
if pool.empty: pool=cand.copy()
R1=pool.sort_values('score',ascending=False).iloc[0]
display(pool.sort_values('score',ascending=False).head(10))
seed=np.array([R1.z_contact,R1.y_contact,R1.x_contact],float); seed_lps=np.array([R1.lps_x,R1.lps_y,R1.lps_z])
print('Selected R1-like candidate:',R1.to_dict())


In [ ]:
z,y,x=np.round(seed).astype(int)
fig,axs=plt.subplots(1,3,figsize=(15,5))
axs[0].imshow(source[z],cmap='gray',vmin=-200,vmax=900); axs[0].contour(aorta[z].astype(float),levels=[.5],linewidths=1.2); axs[0].plot(x,y,'o'); axs[0].set_title(f'R1 z={z}'); axs[0].axis('off')
r=75; y0c,y1c=max(0,y-r),min(source.shape[1],y+r); x0c,x1c=max(0,x-r),min(source.shape[2],x+r)
axs[1].imshow(source[z,y0c:y1c,x0c:x1c],cmap='gray',vmin=-200,vmax=900); axs[1].contour(aorta[z,y0c:y1c,x0c:x1c].astype(float),levels=[.5],linewidths=1.2); axs[1].plot(x-x0c,y-y0c,'o'); axs[1].set_title('R1 close-up'); axs[1].axis('off')
dz=max(1,int(round(3/spacing_xyz[2]))); mip=np.max(source[max(0,z-dz):min(source.shape[0],z+dz+1)],axis=0)
axs[2].imshow(mip,cmap='gray',vmin=-200,vmax=900); axs[2].plot(x,y,'o'); axs[2].set_title('Axial MIP ±3 mm'); axs[2].axis('off')
plt.tight_layout(); p=OUTDIR/'02_R1_neighborhood.png'; fig.savefig(p,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig); print('Saved:',p)


## 5. Build a local vessel-likelihood cost volume around R1


In [ ]:
margin_mm=np.array([18.,28.,28.]); mv=np.ceil(margin_mm/spacing_zyx).astype(int); sz,sy,sx=np.round(seed).astype(int)
z0=max(0,sz-mv[0]); z1=min(source.shape[0],sz+mv[0]+1); y0=max(0,sy-mv[1]); y1=min(source.shape[1],sy+mv[1]+1); x0=max(0,sx-mv[2]); x1=min(source.shape[2],sx+mv[2]+1)
roi=source[z0:z1,y0:y1,x0:x1].astype(np.float32); ar=aorta[z0:z1,y0:y1,x0:x1]
sl=np.array([seed[0]-z0,seed[1]-y0,seed[2]-x0])
sm=ndi.gaussian_filter(roi,sigma=np.maximum(.5,.6/spacing_zyx))
v=frangi(sm,sigmas=range(1,4),black_ridges=False); v=np.nan_to_num(v)
v99=np.percentile(v[v>0],99) if np.any(v>0) else 1; vn=np.clip(v/max(v99,1e-6),0,1)
lo=max(140.,blood_thr*.55); hi=max(lo+100.,med_hu*1.35); ints=np.clip((sm-lo)/(hi-lo),0,1)
support=.58*ints+.42*vn; da=ndi.distance_transform_edt(~ar,sampling=spacing_zyx)
cost=1/(.06+support); cost[ar]+=40; cost[da<.6]+=15; cost[sm<lo]+=12; cost=cost.astype(np.float32)
print('ROI:',roi.shape,' support percentiles:',np.percentile(support,[50,75,90,95,99]))


## 6. Generate 8–22 mm distal endpoint hypotheses and trace the best local path


In [ ]:
zz,yy,xx=np.indices(roi.shape)
dseed=np.sqrt(((zz-sl[0])*spacing_zyx[0])**2+((yy-sl[1])*spacing_zyx[1])**2+((xx-sl[2])*spacing_zyx[2])**2)
thr=np.percentile(support,88)
emask=(dseed>=8)&(dseed<=22)&(da>=2.5)&(sm>=lo)&(support>=thr)
ec=np.argwhere(emask)
rows=[]
for c in ec:
    zc,yc,xc=map(int,c); gl=np.array([zc+z0,yc+y0,xc+x0],float); lps=array_zyx_to_lps(gl); dx=lps[0]-seed_lps[0]
    es=.50*support[zc,yc,xc]+.20*np.tanh(da[zc,yc,xc]/7)+.15*np.tanh(dseed[zc,yc,xc]/15)+.15*np.tanh(max(0,-dx)/8)
    rows.append((float(es),zc,yc,xc))
rows.sort(reverse=True); chosen=[]
for r in rows:
    _,zc,yc,xc=r; p=np.array([zc,yc,xc])*spacing_zyx
    if all(np.linalg.norm(p-q[1])>=3.5 for q in chosen): chosen.append((r,p))
    if len(chosen)>=10: break
eps=[q[0] for q in chosen]
print('Endpoints tested:',len(eps))

def metrics(path):
    p=np.asarray(path,int); hu=roi[tuple(p.T)]; su=support[tuple(p.T)]; dd=da[tuple(p.T)]
    step=np.diff(p.astype(float),axis=0)*spacing_zyx; seg=np.linalg.norm(step,axis=1); L=float(seg.sum())
    if len(step)>=2:
        unit=step/np.maximum(np.linalg.norm(step,axis=1,keepdims=True),1e-6); dots=np.clip(np.sum(unit[:-1]*unit[1:],axis=1),-1,1); turn=float(np.mean(np.arccos(dots)))
    else: turn=np.pi
    outward=float(np.mean(np.diff(dd)>=-.35)) if len(dd)>1 else 0
    return dict(length_mm=L,mean_hu=float(np.mean(hu)),p10_hu=float(np.percentile(hu,10)),
                mean_support=float(np.mean(su)),min_support=float(np.min(su)),
                mean_dist_aorta_mm=float(np.mean(dd)),end_dist_aorta_mm=float(dd[-1]),
                outward_fraction=outward,mean_turn_rad=turn)
routes=[]; start=tuple(np.round(sl).astype(int))
for rank,(es,ez,ey,ex) in enumerate(eps,1):
    try: path,tc=route_through_array(cost,start,(ez,ey,ex),fully_connected=True,geometric=True)
    except Exception: continue
    m=metrics(path)
    if not 7<=m['length_mm']<=30: continue
    rs=.28*np.tanh(m['mean_support']/.55)+.18*np.tanh(m['mean_hu']/350)+.16*np.tanh(m['end_dist_aorta_mm']/8)+.16*m['outward_fraction']+.12*np.exp(-m['mean_turn_rad']/.65)+.10*np.tanh(m['length_mm']/14)
    routes.append(dict(endpoint_rank=rank,endpoint_score=es,route_score=float(rs),total_cost=float(tc),path=np.asarray(path,int),**m))
if not routes: raise RuntimeError('No plausible local continuity route found')
best=max(routes,key=lambda r:r['route_score']); path=best['path']
display(pd.DataFrame([{k:v for k,v in r.items() if k!='path'} for r in routes]).sort_values('route_score',ascending=False))
print('Best:',{k:v for k,v in best.items() if k!='path'})


## 7. Visualize the best local route in orthogonal projections and a dense axial sequence


In [ ]:
pg=path.astype(float); pg[:,0]+=z0; pg[:,1]+=y0; pg[:,2]+=x0
steps=np.diff(pg,axis=0)*spacing_zyx; cum=np.r_[0,np.cumsum(np.linalg.norm(steps,axis=1))]
i10=int(np.argmin(np.abs(cum-10))) if cum[-1]>=8 else None
crop=source[z0:z1,y0:y1,x0:x1]
fig,axs=plt.subplots(1,3,figsize=(16,5))
zm=max(0,int(np.floor(pg[:,0].min()))-2); zM=min(source.shape[0],int(np.ceil(pg[:,0].max()))+3)
mip=np.max(source[zm:zM],axis=0); axs[0].imshow(mip,cmap='gray',vmin=-200,vmax=900); axs[0].plot(pg[:,2],pg[:,1],'-',lw=2); axs[0].plot(pg[0,2],pg[0,1],'o'); axs[0].set_title(f'Axial MIP — {cum[-1]:.1f} mm'); axs[0].axis('off')
cor=np.max(crop,axis=1); axs[1].imshow(cor,cmap='gray',vmin=-200,vmax=900,aspect='auto'); axs[1].plot(path[:,2],path[:,0],'-',lw=2); axs[1].plot(path[0,2],path[0,0],'o'); axs[1].set_title('Local coronal MIP'); axs[1].axis('off')
sag=np.max(crop,axis=2); axs[2].imshow(sag,cmap='gray',vmin=-200,vmax=900,aspect='auto'); axs[2].plot(path[:,1],path[:,0],'-',lw=2); axs[2].plot(path[0,1],path[0,0],'o'); axs[2].set_title('Local sagittal MIP'); axs[2].axis('off')
if i10 is not None:
    axs[0].plot(pg[i10,2],pg[i10,1],'s'); axs[1].plot(path[i10,2],path[i10,0],'s'); axs[2].plot(path[i10,1],path[i10,0],'s')
plt.tight_layout(); p=OUTDIR/'03_best_local_route_orthogonal.png'; fig.savefig(p,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig); print('Saved:',p)


In [ ]:
zshow=np.unique(np.linspace(int(np.floor(pg[:,0].min())),int(np.ceil(pg[:,0].max())),12).round().astype(int))
fig,axs=plt.subplots(3,4,figsize=(16,12))
for ax in axs.ravel(): ax.axis('off')
for ax,zs in zip(axs.ravel(),zshow):
    ax.imshow(source[zs],cmap='gray',vmin=-200,vmax=900); ax.contour(aorta[zs].astype(float),levels=[.5],linewidths=1)
    j=int(np.argmin(np.abs(pg[:,0]-zs)))
    if abs(pg[j,0]-zs)<=2: ax.plot(pg[j,2],pg[j,1],'o'); ax.set_title(f'z={zs}, s≈{cum[j]:.1f} mm')
    else: ax.set_title(f'z={zs}')
    ax.axis('off')
plt.tight_layout(); p=OUTDIR/'04_dense_axial_route_sequence.png'; fig.savefig(p,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig); print('Saved:',p)


## 8. Conservative quantitative interpretation


In [ ]:
checks={
 'length_8mm_or_more':best['length_mm']>=8,
 'mean_HU_180_or_more':best['mean_hu']>=180,
 'p10_HU_100_or_more':best['p10_hu']>=100,
 'mean_support_0.30_or_more':best['mean_support']>=.30,
 'end_away_from_aorta_3mm_or_more':best['end_dist_aorta_mm']>=3,
 'mostly_outward':best['outward_fraction']>=.65,
 'not_excessively_jagged':best['mean_turn_rad']<=1.10,
}
n=sum(checks.values()); status='PASS-CANDIDATE' if n>=6 else ('REVIEW' if n>=4 else 'FAIL')
summary=pd.DataFrame([{'status':status,**checks,**{k:v for k,v in best.items() if k!='path'}}])
display(summary); summary.to_csv(OUTDIR/'R1_continuity_summary.csv',index=False)
print('AUTOMATED STATUS:',status)
print('Please send 02_R1_neighborhood.png, 03_best_local_route_orthogonal.png, 04_dense_axial_route_sequence.png, and the summary row.')
